In [1]:
from pathlib import Path
import pandas as pd

DATA = Path.home() / "Desktop/mimic-iv-clinical-database-demo-2/hosp"
DATA.exists()

True

## Get potassium lab data

In [ ]:
# load labevents.csv
labs = pd.read_csv(
    DATA / "labevents.csv",
    usecols=["subject_id", "hadm_id", "itemid", "charttime", "valuenum"],
    parse_dates=["charttime"],
)
print(labs.shape)
labs.head()

(107727, 5)


,subject_id,hadm_id,itemid,charttime,valuenum
0,10014354,29600294.0,51277,2148-08-16,15.40
1,10014354,29600294.0,51279,2148-08-16,3.35
2,10014354,29600294.0,52172,2148-08-16,49.70
3,10014354,29600294.0,51301,2148-08-16,20.30
4,10014354,29600294.0,51249,2148-08-16,31.10


In [ ]:
# filter for potassium blood test and non-null values
is_k = labs["itemid"].isin([50971, 50822, 52610, 52452])   # potassium blood test
has_value = labs["valuenum"].notna()                       # have value
k = labs[ is_k & has_value ]                               # both conditions pass
len(k)

3321

In [ ]:
# missing hadm_id
k["hadm_id"].isna().sum()

np.int64(602)

## Match with admission data

In [6]:
adm = pd.read_csv(
    DATA / "admissions.csv",
    usecols=["subject_id", "hadm_id", "admittime", "dischtime", "edregtime"],
    parse_dates=["admittime", "dischtime", "edregtime"],
)
adm.head()

,subject_id,hadm_id,admittime,dischtime,edregtime
0,10004235,24181354,2196-02-24 14:38:00,2196-03-04 14:02:00,2196-02-24 12:15:00
1,10009628,25926192,2153-09-17 17:08:00,2153-09-25 13:20:00,NaT
2,10018081,23983182,2134-08-18 02:02:00,2134-08-23 19:35:00,2134-08-17 16:24:00
3,10006053,22942076,2111-11-13 23:39:00,2111-11-15 17:20:00,NaT
4,10031404,21606243,2113-08-04 18:46:00,2113-08-06 20:57:00,NaT


In [7]:
# check missing hadm_id
missing = k[k["hadm_id"].isna()].drop(columns="hadm_id").copy()
missing["lab_row"] = range(len(missing))
len(missing)

602

In [8]:
# matching candidates: inner join on subject_id
cand = missing.merge(adm, on="subject_id", how="inner")
print(len(missing), len(cand))

602 4666


In [ ]:
# filter for candidates that are in the hospital stay
in_stay = (cand["admittime"] <= cand["charttime"]) & (cand["charttime"] <= cand["dischtime"])
matched = cand[in_stay]
matched["lab_row"].nunique()

15

## Check unmatched blood tests

In [11]:
unmatched = missing[~missing["lab_row"].isin(matched["lab_row"])]
len(unmatched)

587

In [12]:
# calculate blood test hours before admission
cand["hours_before_admit"] = (cand["admittime"] - cand["charttime"]).dt.total_seconds() / 3600

In [15]:
# filter for candidates that are not in the hospital stay
c1 = cand[ ~cand["lab_row"].isin(matched["lab_row"]) ]
# filter for candidates that are within 24 hours before admission
c2 = c1[ c1["hours_before_admit"].between(0, 24) ]
# count unique lab rows
c2["lab_row"].nunique()

183

In [16]:
c2[["charttime", "edregtime", "admittime", "hours_before_admit"]].head(10)

,charttime,edregtime,admittime,hours_before_admit
0,2160-11-08 00:00:00,2160-11-07 19:15:00,2160-11-08 04:16:00,4.266667
77,2115-11-08 13:00:00,2115-11-08 12:02:00,2115-11-08 13:54:00,0.900000
133,2149-09-17 10:00:00,2149-09-17 09:08:00,2149-09-17 22:54:00,12.900000
230,2117-10-25 16:00:00,2117-10-25 15:49:00,2117-10-25 22:22:00,6.366667
234,2148-01-25 19:00:00,2148-01-25 18:46:00,2148-01-25 22:58:00,3.966667
282,2144-01-19 06:00:00,2144-01-18 11:53:00,2144-01-19 12:07:00,6.116667
300,2176-08-11 10:00:00,2176-08-11 08:36:00,2176-08-11 15:17:00,5.283333
338,2149-09-20 07:00:00,2149-09-20 05:50:00,2149-09-20 14:30:00,7.500000
374,2137-10-12 18:00:00,NaT,2137-10-12 22:43:00,4.716667
384,2149-03-04 21:00:00,2149-03-04 20:24:00,2149-03-04 23:14:00,2.233333


## Add in ED window


In [17]:
cand["window_start"] = cand["admittime"]                                       # all fill in admittime
ed_earlier = cand["edregtime"].notna() & (cand["edregtime"] < cand["admittime"])  # check if edregtime is earlier than admittime
cand.loc[ed_earlier, "window_start"] = cand.loc[ed_earlier, "edregtime"]       # change the window_start to edregtime

In [18]:
# filter for candidates that are within the window
in_window = (cand["window_start"] <= cand["charttime"]) & (cand["charttime"] <= cand["admittime"])

In [20]:
ed_only = cand[ in_window & ed_earlier ]   # only has edregtime and earlier than admittime
ed_only = ed_only[ ~ed_only["lab_row"].isin(matched["lab_row"]) ]   # remove those already matched
ed_only["lab_row"].nunique()

170

In [22]:
ed_adm = adm["edregtime"].notna() & (adm["edregtime"] < adm["admittime"])
print(ed_adm.sum(), len(adm))

181 275


## Export JSON

In [ ]:
Path("data").mkdir(exist_ok=True)                        # 建立 data 資料夾
# filter for a specific hadm_id and sort by charttime
one = k[k["hadm_id"] == 20297618].sort_values("charttime")
one[["charttime", "valuenum"]].to_json(
    "data/k_20297618.json", orient="records", date_format="iso"
)
len(one)

19

## Add all other patients


In [26]:
k_all = k[k["hadm_id"].notna()].sort_values(["hadm_id", "charttime"])
k_all[["hadm_id", "charttime", "valuenum"]].to_json(
    "data/labs_all.json", orient="records", date_format="iso"
)
print(len(k_all), k_all["hadm_id"].nunique())

2719 250
